In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lfsr_sequence(seed, taps, n_bits):
    """
    seed : 初期状態 例 [1,0,0,1]
    taps : XORを取る位置 例 [0,3]
    n_bits : 生成ビット数
    """
    state = seed[:]
    seq = []

    for _ in range(n_bits):
        out = state[-1]
        seq.append(out)

        feedback = 0
        for t in taps:
            feedback ^= state[t]

        state = [feedback] + state[:-1]

    return np.array(seq)

def cyclic_shift(seq, shift):
    return np.roll(seq, shift)

def bit_to_pm1(bits):
    """
    0,1 を +1,-1 に変換
    相関計算を見やすくする
    """
    return np.where(bits == 0, 1, -1)

def correlation_all_phases(reference, received):
    """
    received を各位相でずらして reference と比較
    """
    ref = bit_to_pm1(reference)
    rec = bit_to_pm1(received)

    corr = []

    for shift in range(len(reference)):
        shifted = np.roll(rec, -shift)
        value = np.sum(ref * shifted)
        corr.append(value)

    return np.array(corr)

# -----------------------------
# 設定
# -----------------------------

m = 4
period = 2**m - 1

# x^4 + x + 1 に対応する例
seed = [1, 0, 0, 0]
taps = [0, 3]

# LFSR系列生成
pn = lfsr_sequence(seed, taps, period)

# 受信側では、送信系列が何ビットかずれて見える
true_shift = 5
received = cyclic_shift(pn, true_shift)

# 相関計算
corr = correlation_all_phases(pn, received)

estimated_shift = np.argmax(corr)

print("送信PN系列:")
print("".join(map(str, pn)))

print("\n受信系列:")
print("".join(map(str, received)))

print("\n本当のずれ:", true_shift)
print("推定されたずれ:", estimated_shift)
print("最大相関値:", corr[estimated_shift])

# -----------------------------
# グラフ表示
# -----------------------------

plt.figure(figsize=(10, 4))
plt.stem(range(period), corr)
plt.axvline(estimated_shift, linestyle="--", label=f"peak shift = {estimated_shift}")
plt.title("LFSR PN系列の同期補足：全位相相関")
plt.xlabel("位相シフト")
plt.ylabel("相関値")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()